# Retail Store Operations — Pre-Filled Configuration
### Ready-to-Run Config for the Retail Use Case

This is a **pre-configured** version of `00_Industry_Config` specifically for the
**Retail Store Manager Documentation Burden** demo. All table lists, artifact names,
and data paths are hardcoded from the existing 23-CSV retail dataset.

**Use this instead of `00_Industry_Config` if you want to skip auto-discovery and run
directly with the known retail tables.**

---

### Data Summary
- **6 Dimensions** — Store Managers, Stores, Districts, Report Types, Product Categories, Vendor Partners
- **6 Batch Facts** — Compliance Doc Events, Inventory Audits, Manager Wellness, Audit Quality, Customer Satisfaction, Scheduling Events
- **6 Event Facts** → Eventhouse — POS Transactions, Inventory Scans, Store Incidents, Shift Handoffs, Retail System Clicks, Associate Presence
- **5 Streams** → Eventhouse — POS Transactions, Foot Traffic, Inventory Scans, Store Incidents, Associate Location

### Ontology
- **6 Entity Types:** StoreManager, Store, District, ReportType, ProductCategory, VendorPartner
- **5 Relationship Types:** manages_store, belongs_to_district, sells_category, served_by_vendor, files_report
- **6 Contextualizations:** ComplianceDocEvent, POSTransaction, InventoryScan, StoreIncident, ShiftHandoff, AssociatePresence

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INDUSTRY SETTING
# ════════════════════════════════════════════════════════════════════════

INDUSTRY       = "retail"
INDUSTRY_LABEL = "Retail"

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# Update these to match your Fabric workspace artifact names.

LAKEHOUSE_NAME      = "RetailLakehouse"
WAREHOUSE_NAME      = "Retail_Datawarehouse"
EVENTHOUSE_NAME     = "retail_rt_store"
ONTOLOGY_NAME       = "RetailOpsOntology"
DATA_AGENT_NAME     = "RetailQA"
OPS_AGENT_NAME      = "RetailDocBurden"
SEMANTIC_MODEL_NAME = "Retail_Ops_Model"

print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA PATHS & EVENTHOUSE CONNECTION
# ════════════════════════════════════════════════════════════════════════

# CSV files location in Lakehouse Files area
CSV_BASE_PATH = "/lakehouse/default/Files/retail_store_operations/data"

# Schemas
LAKEHOUSE_SCHEMA = "dbo"
WAREHOUSE_SCHEMA = "dbo"

# ── Eventhouse Connection ───────────────────────────────────────────────
# Fill these in from your Fabric workspace
EVENTHOUSE_CLUSTER_URI = ""   # e.g. "https://trd-xxxxx.z5.kusto.fabric.microsoft.com"
EVENTHOUSE_DATABASE    = ""   # e.g. "retail_rt_store"

# Ontology package path
ONTOLOGY_PACKAGE_PATH = "/lakehouse/default/Files/retail_ops_ontology_package.iq"

print(f"CSV source:     {CSV_BASE_PATH}")
print(f"LH schema:      {LAKEHOUSE_SCHEMA}")
print(f"WH schema:      {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:     {EVENTHOUSE_CLUSTER_URI or '(fill in your cluster URI)'}")
print(f"Ontology pkg:   {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RETAIL TABLE DEFINITIONS (pre-filled, no auto-discovery needed)
# ════════════════════════════════════════════════════════════════════════

import os, json

# ── 6 Dimension Tables → Lakehouse + Warehouse ──────────────────────────
DIM_TABLES = [
    "dim_store_managers",
    "dim_stores",
    "dim_districts",
    "dim_report_types",
    "dim_product_categories",
    "dim_vendor_partners",
]

# ── 6 Batch Fact Tables → Lakehouse + Warehouse ─────────────────────────
FACT_TABLES_BATCH = [
    "fact_compliance_doc_events",
    "fact_inventory_audits",
    "fact_manager_wellness",
    "fact_audit_quality",
    "fact_customer_satisfaction",
    "fact_scheduling_events",
]

# ── 6 Event Fact Tables → Eventhouse (KQL) ──────────────────────────────
FACT_TABLES_EVENT = [
    "fact_pos_transactions",
    "fact_inventory_scans",
    "fact_store_incidents",
    "fact_shift_handoffs",
    "fact_retail_system_clicks",
    "fact_associate_presence",
]

# ── 5 Streaming Tables → Eventhouse (KQL) ───────────────────────────────
STREAM_TABLES = [
    "stream_pos_transactions",
    "stream_foot_traffic",
    "stream_inventory_scans",
    "stream_store_incidents",
    "stream_associate_location",
]

# ── Combined Target Lists ───────────────────────────────────────────────
LAKEHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH
WAREHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES

# ── KQL Table Name Mapping (CSV name → KQL table name) ─────────────────
# Event facts strip the 'fact_' prefix; streams strip 'stream_' prefix
EVENTHOUSE_KQL_NAMES = {
    "fact_pos_transactions":       "pos_transactions",
    "fact_inventory_scans":        "inventory_scans",
    "fact_store_incidents":        "store_incidents",
    "fact_shift_handoffs":         "shift_handoffs",
    "fact_retail_system_clicks":   "retail_system_clicks",
    "fact_associate_presence":     "associate_presence",
    "stream_pos_transactions":     "pos_transactions_rt",
    "stream_foot_traffic":         "foot_traffic",
    "stream_inventory_scans":      "inventory_scans_rt",
    "stream_store_incidents":      "store_incidents_rt",
    "stream_associate_location":   "associate_location",
}

print(f"{'='*60}")
print(f"RETAIL TABLE INVENTORY")
print(f"{'='*60}")
print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")
print(f"\nFact tables — Batch ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")
print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\n{'─'*60}")
print(f"Lakehouse target:  {len(LAKEHOUSE_TABLES)} tables (12 Delta tables)")
print(f"Warehouse target:  {len(WAREHOUSE_TABLES)} tables (18 SQL tables)")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables (11 KQL tables)")
print(f"Total:             23 CSV files")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPECTED ROW COUNTS (for validation in downstream notebooks)
# ════════════════════════════════════════════════════════════════════════

EXPECTED_ROW_COUNTS = {
    # Dimensions
    "dim_store_managers":              25,
    "dim_stores":                      30,
    "dim_districts":                    5,
    "dim_report_types":                18,
    "dim_product_categories":          15,
    "dim_vendor_partners":             12,
    # Batch Facts
    "fact_compliance_doc_events":     180,
    "fact_inventory_audits":           90,
    "fact_manager_wellness":           25,
    "fact_audit_quality":              60,
    "fact_customer_satisfaction":      30,
    "fact_scheduling_events":         120,
    # Event Facts
    "fact_pos_transactions":          500,
    "fact_inventory_scans":           250,
    "fact_store_incidents":            35,
    "fact_shift_handoffs":             60,
    "fact_retail_system_clicks":      300,
    "fact_associate_presence":        400,
    # Streams
    "stream_pos_transactions":         80,
    "stream_foot_traffic":             80,
    "stream_inventory_scans":          80,
    "stream_store_incidents":           80,
    "stream_associate_location":        80,
}

total_lakehouse = sum(EXPECTED_ROW_COUNTS[t] for t in LAKEHOUSE_TABLES)
total_eventhouse = sum(EXPECTED_ROW_COUNTS[t] for t in EVENTHOUSE_TABLES)
print(f"Expected Lakehouse rows:  {total_lakehouse:,}")
print(f"Expected Eventhouse rows: {total_eventhouse:,}")
print(f"Expected total rows:      {sum(EXPECTED_ROW_COUNTS.values()):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SCHEMA INSPECTION HELPER
# ════════════════════════════════════════════════════════════════════════

def preview_table(table_name, base_path=CSV_BASE_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"\n{'─'*60}")
    print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
    print(f"{'─'*60}")
    df.printSchema()
    df.show(rows, truncate=False)
    return df

# Uncomment to preview specific tables:
# preview_table("dim_store_managers")
# preview_table("fact_pos_transactions")
# preview_table("stream_foot_traffic")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(f"\n{'═'*60}")
print(f"✅ RETAIL CONFIG READY")
print(f"{'═'*60}")
print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))
print(f"\nThis config is imported by downstream notebooks via:")
print(f"  %run ./Retail_Config")